In [1]:
import mlflow
from pathlib import Path
from mlflow.tracking import MlflowClient

In [2]:
cwd = Path.cwd()
project_root = cwd.parent

In [3]:
# Tracking backend (SQLite at project root)
mlflow.set_tracking_uri(f'sqlite:///{project_root}/mlflow.db')

# Artifact root — separate folder at project root
artifact_root = f'file://{project_root}/mlartifacts'

In [4]:
experiment_names = [
    "M6_baselines",
    "Pass2_experiments",
    "Regime_B_baselines",
    "M7_xgb_tuning",
    "M12_mlp",
    "M13_final_selection",
]

client = MlflowClient()
for name in experiment_names:
    # create_experiment fails if it exists, so guard it
    existing = client.get_experiment_by_name(name)
    if existing is None:
        exp_id = mlflow.create_experiment(
            name=name,
            artifact_location=f"{artifact_root}/{name}"
        )
        print(f"created: {name} ({exp_id})")
    else:
        print(f"exists: {name} ({existing.experiment_id})")

2026/07/07 19:23:06 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/07 19:23:06 INFO mlflow.store.db.utils: Updating database tables


created: M6_baselines (1)
created: Pass2_experiments (2)
created: Regime_B_baselines (3)
created: M7_xgb_tuning (4)
created: M12_mlp (5)
created: M13_final_selection (6)


In [5]:
for name in experiment_names:
    exp = client.get_experiment_by_name(name)
    print(f"{name}: {exp.artifact_location}")

M6_baselines: file:///Users/ak007/SML/Credit-Risk-Default-Prediction-System/mlartifacts/M6_baselines
Pass2_experiments: file:///Users/ak007/SML/Credit-Risk-Default-Prediction-System/mlartifacts/Pass2_experiments
Regime_B_baselines: file:///Users/ak007/SML/Credit-Risk-Default-Prediction-System/mlartifacts/Regime_B_baselines
M7_xgb_tuning: file:///Users/ak007/SML/Credit-Risk-Default-Prediction-System/mlartifacts/M7_xgb_tuning
M12_mlp: file:///Users/ak007/SML/Credit-Risk-Default-Prediction-System/mlartifacts/M12_mlp
M13_final_selection: file:///Users/ak007/SML/Credit-Risk-Default-Prediction-System/mlartifacts/M13_final_selection


In [6]:
import json

In [7]:
def log_historical_run(
    experiment_name: str,
    run_name: str,
    model_dir: Path,
    tags: dict[str, str],
    fn_cost: int,
    fp_cost: int,
    metrics_file_name: str = 'metrics',
    additional_params: dict[str, object] = None
):
    mlflow.set_experiment(experiment_name)
    
    with mlflow.start_run(run_name=run_name):
        
        # set tags
        mlflow.set_tags(tags)
        
        # reading the metrics and threshold
        with open(model_dir / f'{metrics_file_name}.json') as f:
            metrics = json.load(f)
            
        flat_metrics = {}
        for split in ['train', 'val', 'test']:
            if split not in metrics:
                continue
            
            for metric_name, value in metrics[split].items():
                if metric_name == 'confusion_matrix':
                    continue
                key = f"{split}_{metric_name.lower().replace('-', '_')}"
                
                flat_metrics[key] = value
                
        mlflow.log_metrics(flat_metrics)
        
        # logging threshold, fncost, fpcost
        params_from_threshold = {
            'threshold': metrics['threshold'],
            'fn_cost': fn_cost,
            'fp_cost': fp_cost,
        }
        
        mlflow.log_params(params_from_threshold)
        
        # adding additional params related to the model
        if additional_params:
            mlflow.log_params(additional_params)
            
        # logging artifacts
        for file in model_dir.iterdir():
            if file.is_file():
                mlflow.log_artifact(str(file))

In [8]:
# M6_baselines
# baseline_lr
model_path = project_root / 'models' / 'baseline_lr'
tags = {
    "milestone": "M6",
    "regime": "A",
    "feature_pass": "pass_1",
    "tuned": "false",
}
additional_params={
    "model_family": "logistic_regression",
    "max_iter": 100,
    'C': 1.0,
    'regularization': 'L2'
}
log_historical_run(
    experiment_name='M6_baselines',
    run_name='LR_baseline_regime_A',
    model_dir=model_path,
    tags=tags,
    additional_params=additional_params,
    fn_cost=9000,
    fp_cost=2000
)

In [9]:
# M6_baselines
# baseline_rf
model_path = project_root / 'models' / 'baseline_rf'
tags = {
    "milestone": "M6",
    "regime": "A",
    "feature_pass": "pass_1",
    "tuned": "false",
}
additional_params={
    "model_family": "random_forest",
    "n_estimators": 100,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "random_state": 42,
}
log_historical_run(
    experiment_name='M6_baselines',
    run_name='RF_baseline_regime_A',
    model_dir=model_path,
    tags=tags,
    additional_params=additional_params,
    fn_cost=10000,
    fp_cost=2000
)

In [10]:
# M6_baselines
# baseline_rf
model_path = project_root / 'models' / 'baseline_xgb'
tags = {
    "milestone": "M6",
    "regime": "A",
    "feature_pass": "pass_1",
    "tuned": "false",
}
additional_params={
    "model_family": "xgboost",
    "n_estimators": 100,    
    "max_depth": 6,         
    "learning_rate": 0.3,   
    "subsample": 1.0,       
    "colsample_bytree": 1.0,
    "min_child_weight": 1,  
    "reg_alpha": 0,         
    "reg_lambda": 1,        
}
log_historical_run(
    experiment_name='M6_baselines',
    run_name='XGB_baseline_regime_A',
    model_dir=model_path,
    tags=tags,
    additional_params=additional_params,
    fn_cost=10000,
    fp_cost=2000
)

In [11]:
# M6_baselines
# baseline_lr

experiment_name = "Pass2_experiments"
run_name = 'LR_after_FE_metrics'
model_path = project_root / 'models' / 'after_FE'
tags = {
    "milestone": "M6",
    "regime": "B",
    "feature_pass": "pass_2",
    "tuned": "false",
}
additional_params={
    "model_family": "logistic_regression",
    "max_iter": 100,
    'C': 1.0,
    'regularization': 'L2'
}
log_historical_run(
    experiment_name=experiment_name,
    run_name=run_name,
    model_dir=model_path,
    tags=tags,
    additional_params=additional_params,
    metrics_file_name='after_fe_lr',
    fn_cost=10000,
    fp_cost=2000
)

In [12]:
# M6_baselines
# baseline_lr

experiment_name = "Pass2_experiments"
run_name = 'RF_after_FE_metrics'
model_path = project_root / 'models' / 'after_FE'
tags = {
    "milestone": "M6",
    "regime": "B",
    "feature_pass": "pass_2",
    "tuned": "false",
}
additional_params={
    "model_family": "random_forest",
    "n_estimators": 100,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "random_state": 42,
}
log_historical_run(
    experiment_name=experiment_name,
    run_name=run_name,
    model_dir=model_path,
    tags=tags,
    additional_params=additional_params,
    metrics_file_name='after_fe_rf',
    fn_cost=10000,
    fp_cost=2000
)

In [13]:
# M6_baselines
# baseline_lr

experiment_name = "Pass2_experiments"
run_name = 'XGB_after_FE_metrics'
model_path = project_root / 'models' / 'after_FE'
tags = {
    "milestone": "M6",
    "regime": "B",
    "feature_pass": "pass_2",
    "tuned": "false",
}
additional_params={
    "model_family": "xgboost",
    "n_estimators": 100,    
    "max_depth": 6,         
    "learning_rate": 0.3,   
    "subsample": 1.0,       
    "colsample_bytree": 1.0,
    "min_child_weight": 1,  
    "reg_alpha": 0,         
    "reg_lambda": 1,        
}
log_historical_run(
    experiment_name=experiment_name,
    run_name=run_name,
    model_dir=model_path,
    tags=tags,
    additional_params=additional_params,
    metrics_file_name='after_fe_xgb',
    fn_cost=10000,
    fp_cost=2000
)

In [14]:
# M7_xgb_tuning

experiment_name = "M7_xgb_tuning"
run_name = 'XGB_Tuning'
model_path = project_root / 'models' / 'tuned_xgb'
tags = {
    "milestone": "M7",
    "regime": "B",
    "feature_pass": "pass_1",
    "tuned": "True",
}
additional_params={
    "model_family": "xgboost",
    "n_estimators": 895,
    "max_depth": 7,
    "learning_rate": 0.011425456812654167,
    "subsample": 0.5833351412852172,
    "colsample_bytree": 0.7668598048854729,
    "min_child_weight": 10,
    "reg_alpha": 1.0261238904989621e-05,
    "reg_lambda": 0.0006454118557372465        
}
log_historical_run(
    experiment_name=experiment_name,
    run_name=run_name,
    model_dir=model_path,
    tags=tags,
    additional_params=additional_params,
    fn_cost=10000,
    fp_cost=2000
)

In [15]:
# M12_mlp

experiment_name = "M12_mlp"
run_name = 'MLP_Training'
model_path = project_root / 'models' / 'mlp'
tags = {
    "milestone": "M12",
    "regime": "B",
    "feature_pass": "pass_1",
    "tuned": "False",
}
additional_params={
    "n_epochs": 30,
    "batch_size": 2048,
    "learning_rate": 0.001,
    "optimizer": "Adam",
    "loss": "BCEWithLogitsLoss",
    "cv_mean_pr_auc": 0.298,
    "framework": "pytorch",
    "framework_version": "2.12.1",
    "input_dim": 147,
    "hidden_dim": [
        128,
        64
    ],
    "dropout": 0.2
}
log_historical_run(
    experiment_name=experiment_name,
    run_name=run_name,
    model_dir=model_path,
    tags=tags,
    additional_params=additional_params,
    fn_cost=10000,
    fp_cost=2000
)

In [16]:
exp = client.get_experiment_by_name("Regime_B_baselines")
client.delete_experiment(exp.experiment_id)

In [18]:
experiment_name = "M13_final_selection"
run_name = "Ensemble_avg_LR_XGB_MLP"
model_path = project_root / "models" / "ensemble"

tags = {
    "milestone": "M13",
    "regime": "B",
    "feature_pass": "pass_1",
    "tuned": "false",
    "selected": "false",
    "notes": "LR + tuned_XGB + MLP averaged probabilities. Reference for M13 decision matrix.",
}

additional_params = {
    "model_family": "ensemble_avg",
    "components": "LR + tuned_XGB + MLP",
    "aggregation": "mean_probability",
    "corr_lr_xgb": 0.924,
    "corr_lr_mlp": 0.871,
    "corr_xgb_mlp": 0.895,
}

log_historical_run(
    experiment_name=experiment_name,
    run_name=run_name,
    model_dir=model_path,
    tags=tags,
    additional_params=additional_params,
    metrics_file_name="metrics",
    fn_cost=10000,
    fp_cost=2000,
)

In [19]:
# M13_final_selection
# Selected production model — tuned XGBoost from M7, tagged as chosen

experiment_name = "M13_final_selection"
run_name = "SELECTED_tuned_xgb_production"
model_path = project_root / "models" / "tuned_xgb"

tags = {
    "milestone": "M13",
    "regime": "B",
    "feature_pass": "pass_1",
    "tuned": "true",
    "selected": "true",
    "notes": "Final production model. Selected on ECOA interpretability + operational maturity grounds. Ensemble was marginally better (+0.002 test PR-AUC) but rejected due to 3x inference cost and tripled operational surface.",
}

additional_params = {
    # Model family and hyperparameters (same as M7 tuned XGB)
    "model_family": "xgboost",
    "n_estimators": 895,
    "max_depth": 7,
    "learning_rate": 0.011,
    "subsample": 0.58,
    "colsample_bytree": 0.77,
    "min_child_weight": 10,
    "reg_alpha": 0.0,
    "reg_lambda": 0.0,
    "tuning_method": "optuna",
    "tuning_n_trials": 50,
    
    # M13 operational measurements — production spec
    "latency_single_median_ms": 1.43,
    "latency_single_p95_ms": 1.77,
    "latency_batch1000_median_ms": 4.20,
    "latency_batch1000_per_row_ms": 0.004,
    "model_size_mb": 5.4,
    
    # Robustness
    "perturbation_2pct_pr_auc_delta": -0.003,
    "perturbation_5pct_pr_auc_delta": -0.009,
    "perturbation_10pct_pr_auc_delta": -0.018,
    "directional_dti_pass": "true",
    "directional_annual_inc_pass": "true",
}

log_historical_run(
    experiment_name=experiment_name,
    run_name=run_name,
    model_dir=model_path,
    tags=tags,
    additional_params=additional_params,
    metrics_file_name="metrics",
    fn_cost=10000,
    fp_cost=2000,
)

In [20]:
experiment_name = "M13_final_selection"
run_name = "Candidate_mlp"
model_path = project_root / "models" / "mlp"

tags = {
    "milestone": "M13",
    "regime": "B",
    "feature_pass": "pass_1",
    "tuned": "false",
    "selected": "false",
    "notes": "M13 candidate. Rejected on interpretability (weaker SHAP support) and operational maturity grounds despite marginal latency/size wins.",
}

additional_params = {
    "model_family": "mlp",
    "hidden_layers": "[128, 64]",
    "dropout": 0.2,
    "batchnorm": "false",
    "batch_size": 2048,
    "learning_rate": 1e-3,
    "epochs": 30,
    "optimizer": "adam",
    "loss": "BCEWithLogitsLoss",
    
    # M13 operational measurements
    "latency_single_median_ms": 1.21,
    "latency_single_p95_ms": 1.54,
    "latency_batch1000_median_ms": 2.42,
    "latency_batch1000_per_row_ms": 0.002,
    "model_size_mb": 0.12,
    
    # Robustness
    "perturbation_2pct_pr_auc_delta": -0.001,
    "perturbation_5pct_pr_auc_delta": -0.006,
    "perturbation_10pct_pr_auc_delta": -0.021,
    "directional_dti_pass": "true",
    "directional_annual_inc_pass": "true",
}

log_historical_run(
    experiment_name=experiment_name,
    run_name=run_name,
    model_dir=model_path,
    tags=tags,
    additional_params=additional_params,
    metrics_file_name="metrics",
    fn_cost=10000,
    fp_cost=2000,
)